## План занятия:
- Вспомнить, что было
- Новый способ создания и изменения признаков
    - apply
    - replace
- Как переименовать признак/индекс
    - rename
    - set_axis
    - columns/index

Чтобы получить доступ ко всем волшебным методам в pandas, для начала нужно его импортировать себе в ноутбук.

Импортируем pandas и даем ему псевдоним `pd`, чтобы при обращении не писать всё название пакета.

In [ ]:
import pandas as pd

## Задача на сегодня

Давайте возьмем данные с google drive про покупателей с предыдущего занятия: https://drive.google.com/file/d/1BoUIRLFzs-_eb7M6Zn-ic1z-MFUfr4I9

In [ ]:
!wget 'https://drive.google.com/uc?id=1BoUIRLFzs-_eb7M6Zn-ic1z-MFUfr4I9' -O data.csv

--2022-07-20 13:15:21--  https://drive.google.com/uc?id=1BoUIRLFzs-_eb7M6Zn-ic1z-MFUfr4I9
Resolving drive.google.com (drive.google.com)... 108.177.119.139, 108.177.119.102, 108.177.119.100, ...
Connecting to drive.google.com (drive.google.com)|108.177.119.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://doc-10-c0-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/ktu2n1qta0gbb6nva9dvt400p6k1pvql/1658322900000/14904333240138417226/*/1BoUIRLFzs-_eb7M6Zn-ic1z-MFUfr4I9?uuid=053a6bb3-3fb9-4cc5-a72a-e357e9f85415 [following]
--2022-07-20 13:15:22--  https://doc-10-c0-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/ktu2n1qta0gbb6nva9dvt400p6k1pvql/1658322900000/14904333240138417226/*/1BoUIRLFzs-_eb7M6Zn-ic1z-MFUfr4I9?uuid=053a6bb3-3fb9-4cc5-a72a-e357e9f85415
Resolving doc-10-c0-docs.googleusercontent.com (doc-10-c0-docs.googleusercontent.com)... 172.217.218.132, 2a00:1450:4013:c08::84
Connecting to d

In [ ]:
df = pd.read_csv('data.csv', sep=';')
df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,medium avg check
1,32.0,630.0,11.0,0,57.27,small avg check
2,52.0,2730.0,7.0,1,390.00,medium avg check
3,33.0,552.0,2.0,0,276.00,medium avg check
4,35.0,409.0,5.0,0,81.80,small avg check


## Создание нового признака

### .loc
Уже создавали новый признак, который показывает, к какой группе относится наш покупатель:
- small средний чек (< 150)
- medium средний чек (>= 150, < 600)
- large средний чек (>= 600)

Всё это делали используя .loc. Давайте вспомним, в чем заключалась суть

Для начала создавали пустой новый признак.

In [ ]:
df['check_category'] = 0

df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,0
1,32.0,630.0,11.0,0,57.27,0
2,52.0,2730.0,7.0,1,390.00,0
3,33.0,552.0,2.0,0,276.00,0
4,35.0,409.0,5.0,0,81.80,0


Затем фильтровались по категории small check ('avg_check' < 150) и в признаке check_category проставляли строку 'small avg check'.

In [ ]:
df.loc[df['avg_check'] < 150, 'check_category'] = 'small avg check'

df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,0
1,32.0,630.0,11.0,0,57.27,small avg check
2,52.0,2730.0,7.0,1,390.00,0
3,33.0,552.0,2.0,0,276.00,0
4,35.0,409.0,5.0,0,81.80,small avg check


Тоже самое делали для двух других категорий.

**medium средний чек (>= 150, < 600)**

Так как условие уже довольно внушительное, то давайте его запишим в новую переменную.

In [ ]:
condition = (df['avg_check'] >= 150) & (df['avg_check'] < 600)
df.loc[condition, 'check_category'] = 'medium avg check'
df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,medium avg check
1,32.0,630.0,11.0,0,57.27,small avg check
2,52.0,2730.0,7.0,1,390.00,medium avg check
3,33.0,552.0,2.0,0,276.00,medium avg check
4,35.0,409.0,5.0,0,81.80,small avg check


**large средний чек (>= 600)**

И еще одна категория


Работаем по старой схеме.

In [ ]:
condition = (df['avg_check'] >= 600)
df.loc[condition, 'check_category'] = 'large avg check'

Проверим, что мы всё сделали правильно, и у нас теперь есть три категории.

In [ ]:
df['check_category'].value_counts()

medium avg check    59
small avg check     27
large avg check     14
Name: check_category, dtype: int64

### .apply()

Как можно создать подобный признак, но за меньшее количество строк кода и с более читабельным кодом?

Через метод .apply()

Его суть в том, чтобы применить функцию ко всем значениям в таблице или ко всем значениям в конкретной колонке в таблице. Только это будет гораздо быстрее, чем проходиться в цикле.

Давайте небольшой пример на одной колонке разберем.

In [ ]:
def foo(x):
    print(x)
    return x

df.iloc[:5]['age'].apply(foo)

38.0
32.0
52.0
33.0
35.0


0    38.0
1    32.0
2    52.0
3    33.0
4    35.0
Name: age, dtype: float64

А теперь разберем небольшой пример на всей таблице.

In [ ]:
tmp_df = df.iloc[:5].drop(columns=['check_category', 'avg_check'])
tmp_df

,age,sum,num_purchases,is_interested
0,38.0,741.0,2.0,0
1,32.0,630.0,11.0,0
2,52.0,2730.0,7.0,1
3,33.0,552.0,2.0,0
4,35.0,409.0,5.0,0


Можно в функции apply проходиться по столбцам таблицы, если использовать атрибут axis=0.

In [ ]:
def foo(x):
    print(x)
    print('_'* 30)
    return x

tmp_df.apply(foo, axis=0)

0    38.0
1    32.0
2    52.0
3    33.0
4    35.0
Name: age, dtype: float64
______________________________
0     741.0
1     630.0
2    2730.0
3     552.0
4     409.0
Name: sum, dtype: float64
______________________________
0     2.0
1    11.0
2     7.0
3     2.0
4     5.0
Name: num_purchases, dtype: float64
______________________________
0    0
1    0
2    1
3    0
4    0
Name: is_interested, dtype: int64
______________________________


,age,sum,num_purchases,is_interested
0,38.0,741.0,2.0,0
1,32.0,630.0,11.0,0
2,52.0,2730.0,7.0,1
3,33.0,552.0,2.0,0
4,35.0,409.0,5.0,0


К примеру, можем посчитать отклонение от среднего по столбцам.

In [ ]:
def foo(x):
    mean = x.mean()
    print(x.name, mean)
    return x - mean

tmp_df.apply(foo, axis=0)

age 38.0
sum 1012.4
num_purchases 5.4
is_interested 0.2


,age,sum,num_purchases,is_interested
0,0.0,-271.4,-3.4,-0.2
1,-6.0,-382.4,5.6,-0.2
2,14.0,1717.6,1.6,0.8
3,-5.0,-460.4,-3.4,-0.2
4,-3.0,-603.4,-0.4,-0.2


Или же проходиться по строкам таблицы, если поставить атрибут axis=1.

In [ ]:
def foo(x):
    print(x)
    print('_'* 30)
    return x

tmp_df.apply(foo, axis=1)

age               38.0
sum              741.0
num_purchases      2.0
is_interested      0.0
Name: 0, dtype: float64
______________________________
age               32.0
sum              630.0
num_purchases     11.0
is_interested      0.0
Name: 1, dtype: float64
______________________________
age                52.0
sum              2730.0
num_purchases       7.0
is_interested       1.0
Name: 2, dtype: float64
______________________________
age               33.0
sum              552.0
num_purchases      2.0
is_interested      0.0
Name: 3, dtype: float64
______________________________
age               35.0
sum              409.0
num_purchases      5.0
is_interested      0.0
Name: 4, dtype: float64
______________________________


,age,sum,num_purchases,is_interested
0,38.0,741.0,2.0,0.0
1,32.0,630.0,11.0,0.0
2,52.0,2730.0,7.0,1.0
3,33.0,552.0,2.0,0.0
4,35.0,409.0,5.0,0.0


И посчитать средний чек по каждому покупателю.

In [ ]:
def foo(x):
    print('obj', x.name, end=': ')
    sum = x['sum']
    n_purch = x['num_purchases']

    print(sum, '/', n_purch)
    x['avg_check'] = sum / n_purch
    return x

tmp_df.apply(foo, axis=1)

obj 0: 741.0 / 2.0
obj 1: 630.0 / 11.0
obj 2: 2730.0 / 7.0
obj 3: 552.0 / 2.0
obj 4: 409.0 / 5.0


,age,sum,num_purchases,is_interested,avg_check
0,38.0,741.0,2.0,0.0,370.500000
1,32.0,630.0,11.0,0.0,57.272727
2,52.0,2730.0,7.0,1.0,390.000000
3,33.0,552.0,2.0,0.0,276.000000
4,35.0,409.0,5.0,0.0,81.800000


Вот теперь давайте возвращаемся к исходным данным и реализуем функцию, которая сравнивает средний чек с границами и возвращает соответствующую группу.

In [ ]:
def get_check_group(avg_check):
    if avg_check < 150:
        return 'small avg check'
    elif avg_check >= 150 and avg_check < 600:
        return 'medium avg check'
    elif avg_check > 600:
        return 'large avg check'

А затем эту функцию нужно передать в метод .apply() у датафрейма.

In [ ]:
 df['avg_check'].apply(get_check_group)

0     medium avg check
1      small avg check
2     medium avg check
3     medium avg check
4      small avg check
            ...       
95     small avg check
96    medium avg check
97    medium avg check
98    medium avg check
99     small avg check
Name: avg_check, Length: 100, dtype: object

In [ ]:
df['check_category'] = df['avg_check'].apply(get_check_group)
df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,medium avg check
1,32.0,630.0,11.0,0,57.27,small avg check
2,52.0,2730.0,7.0,1,390.00,medium avg check
3,33.0,552.0,2.0,0,276.00,medium avg check
4,35.0,409.0,5.0,0,81.80,small avg check


А если в вашем наборе данных очень много строк, то apply может выполняться долго, в этом случае можно выводить прогресс бар обхода датафрейма через библиотеку tqdm.

<img src='https://memepedia.ru/wp-content/uploads/2017/04/slowpoke-original.png' width=70>

In [ ]:
import time
from tqdm import tqdm
tqdm.pandas()

def slowpoke(x):
    time.sleep(0.1)
    return x

df.progress_apply(slowpoke, axis=1)

100%|██████████| 100/100 [00:10<00:00,  9.82it/s]


,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,0,370.50,medium avg check
1,32.0,630.0,11.0,0,57.27,small avg check
2,52.0,2730.0,7.0,1,390.00,medium avg check
3,33.0,552.0,2.0,0,276.00,medium avg check
4,35.0,409.0,5.0,0,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,0,108.80,small avg check
96,31.0,745.0,3.0,0,248.33,medium avg check
97,31.0,782.0,4.0,0,195.50,medium avg check
98,38.0,793.0,4.0,0,198.25,medium avg check


### .replace()

Данный метод позволяет сделать замену в каком-то признаке, используя весь датафрейм

In [ ]:
df.replace(
    {'is_interested':
        {0: 'no', 1: 'yes'}
    }
)

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


Или же только pd.Series

In [ ]:
df['is_interested'].replace({0: 'no', 1: 'yes'})

0      no
1      no
2     yes
3      no
4      no
     ... 
95     no
96     no
97     no
98     no
99     no
Name: is_interested, Length: 100, dtype: object

Или же сразу в нескольких признаках

In [ ]:
df.replace({0: 'null', 2: 'two'})

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,two,null,370.5,medium avg check
1,32.0,630.0,11.0,null,57.27,small avg check
2,52.0,2730.0,7.0,1,390.0,medium avg check
3,33.0,552.0,two,null,276.0,medium avg check
4,35.0,409.0,5.0,null,81.8,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,null,108.8,small avg check
96,31.0,745.0,3.0,null,248.33,medium avg check
97,31.0,782.0,4.0,null,195.5,medium avg check
98,38.0,793.0,4.0,null,198.25,medium avg check


Чтобы изменения вступили в силу, нужно указать атрибут inplace=True

In [ ]:
df.replace(
    {'is_interested':
        {0: 'no', 1: 'yes'}
    },
    inplace=True
)

df.head()

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check


## Как переименовать признаки/индексы

### .rename()

Изменить названия признака можно с помощью метода rename

In [ ]:
df.rename({'sum': 'sum_purchases'})

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


Но нужно указывать, какую именно оси хочется изменить, потому что по умолчанию меняются индексы

In [ ]:
df.rename({0: '000'})

,age,sum,num_purchases,is_interested,avg_check,check_category
000,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


А вот columns - это чтобы поменять названия колонок

In [ ]:
df.rename(columns={'sum': 'sum_purchases'})

,age,sum_purchases,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


In [ ]:
df

,age,sum,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


При этом помним про атрибут inplace

In [ ]:
df.rename(columns={'sum': 'sum_purchases'}, inplace=True)
df.head()

,age,sum_purchases,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check


А еще можно передавать функцию, по которой хотите делать переименования

In [ ]:
df.rename(columns=lambda x: x + '_1')

,age_1,sum_purchases_1,num_purchases_1,is_interested_1,avg_check_1,check_category_1
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


In [ ]:
def rename_cols(x):
    return x + '_1'

df.rename(columns=rename_cols)

,age,sum_purchases,num_purchases,is_interested,avg_check,check_category
0_1,38.0,741.0,2.0,no,370.50,medium avg check
1_1,32.0,630.0,11.0,no,57.27,small avg check
2_1,52.0,2730.0,7.0,yes,390.00,medium avg check
3_1,33.0,552.0,2.0,no,276.00,medium avg check
4_1,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95_1,40.0,1088.0,10.0,no,108.80,small avg check
96_1,31.0,745.0,3.0,no,248.33,medium avg check
97_1,31.0,782.0,4.0,no,195.50,medium avg check
98_1,38.0,793.0,4.0,no,198.25,medium avg check


### .set_axis()

Есть метод set_axis, который позволяет передать список новых именования признаков (с axis=1)

In [ ]:
df.set_axis(['client_age', 'sum_purchases', 'num_purchases', 'is_interested',  'avg_check', 'check_category'], axis=1)

,client_age,sum_purchases,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
95,40.0,1088.0,10.0,no,108.80,small avg check
96,31.0,745.0,3.0,no,248.33,medium avg check
97,31.0,782.0,4.0,no,195.50,medium avg check
98,38.0,793.0,4.0,no,198.25,medium avg check


Или же можно переименовать индексы, если указать axis=0 (он же стоит по умолчанию)

In [ ]:
import numpy as np

df.set_axis(np.arange(100, 200), axis=0)

,age,sum_purchases,num_purchases,is_interested,avg_check,check_category
100,38.0,741.0,2.0,no,370.50,medium avg check
101,32.0,630.0,11.0,no,57.27,small avg check
102,52.0,2730.0,7.0,yes,390.00,medium avg check
103,33.0,552.0,2.0,no,276.00,medium avg check
104,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
195,40.0,1088.0,10.0,no,108.80,small avg check
196,31.0,745.0,3.0,no,248.33,medium avg check
197,31.0,782.0,4.0,no,195.50,medium avg check
198,38.0,793.0,4.0,no,198.25,medium avg check


In [ ]:
df

,age,sum_purchases,num_purchases,is_interested,avg_check,check_category
100,38.0,741.0,2.0,no,370.50,medium avg check
101,32.0,630.0,11.0,no,57.27,small avg check
102,52.0,2730.0,7.0,yes,390.00,medium avg check
103,33.0,552.0,2.0,no,276.00,medium avg check
104,35.0,409.0,5.0,no,81.80,small avg check
...,...,...,...,...,...,...
195,40.0,1088.0,10.0,no,108.80,small avg check
196,31.0,745.0,3.0,no,248.33,medium avg check
197,31.0,782.0,4.0,no,195.50,medium avg check
198,38.0,793.0,4.0,no,198.25,medium avg check


И здесь так же есть атрибут inplace

In [ ]:
df.set_axis(np.arange(100, 200), axis=0, inplace=True)

### Атрибуты columns/index

In [ ]:
df.columns

Index(['age', 'sum_purchases', 'num_purchases', 'is_interested', 'avg_check',
       'check_category'],
      dtype='object')

In [ ]:
df.columns = ['client_age', 'sum_purchases', 'num_purchases', 'is_interested',  'avg_check', 'check_category']
df.head()

,client_age,sum_purchases,num_purchases,is_interested,avg_check,check_category
100,38.0,741.0,2.0,no,370.50,medium avg check
101,32.0,630.0,11.0,no,57.27,small avg check
102,52.0,2730.0,7.0,yes,390.00,medium avg check
103,33.0,552.0,2.0,no,276.00,medium avg check
104,35.0,409.0,5.0,no,81.80,small avg check


In [ ]:
df.index

Int64Index([100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
            113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
            126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
            139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151,
            152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164,
            165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177,
            178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190,
            191, 192, 193, 194, 195, 196, 197, 198, 199],
           dtype='int64')

In [ ]:
df.index = np.arange(0, 100)
df.head()

,client_age,sum_purchases,num_purchases,is_interested,avg_check,check_category
0,38.0,741.0,2.0,no,370.50,medium avg check
1,32.0,630.0,11.0,no,57.27,small avg check
2,52.0,2730.0,7.0,yes,390.00,medium avg check
3,33.0,552.0,2.0,no,276.00,medium avg check
4,35.0,409.0,5.0,no,81.80,small avg check


## Что сегодня узнали?

- Новый способ создания и изменения признаков
    - apply
    - replace
- Как переименовать признак/индекс
    - rename
    - set_axis
    - columns/index

**Муррр** ♥